# Import dependencies

In [1]:
import tensorflow as tf
from transformers import TFBertModel, TFBertTokenizer, AutoTokenizer
from tensorflow.keras.layers import Input, Dense, Dropout, Attention
from tensorflow.keras.models import Model
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset

# Prepare Data

In [4]:
file_path = './data.pkl'
data = pd.read_pickle(file_path)
data.head()

,Text,Type,Short Summary
0,دُنیا کی تاریخ زمین اور سمندروں پر ہونے والے ت...,History,سمندر کی گہرائیوں میں پائی جانے والی یہ معدنیا...
1,عام طور پر 5-6 ملی میٹر کی پتھری پیشاب کے ذریع...,Health,تاہم اگر مثانے میں پتھری ہو یا سائز میں چھوٹی ...
2,کوڑے کیسے شروع ہوئے؟\nشروع میں دالوں کو پیس کر...,History,کئی جگہوں پر اس کا ذکر بھی آیا ہے کہ پکوڑے یا ...
3,پپیتا نرم و ملائم جلد رکھنے والا پھل ہے۔یہ ابت...,Nature,پپیتا نرم و ملائم جلد رکھنے والا پھل ہے۔
4,دل کو تقویت دیتا ہے۔قے اور بخار سے نجات دلاتا ...,General,دل کو تقویت دیتا ہے۔


In [6]:
test_data = data[900:]
data = data[0:900]

In [7]:
from transformers import AutoTokenizer, TFBertModel
import tensorflow as tf

tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")
model = TFBertModel.from_pretrained("google-bert/bert-base-uncased")

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.seq_relationship.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.bias', 'cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.bias']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertModel for predictions w

In [8]:
# Function to tokenize the data
def tokenize_and_prepare_masks(data, max_length):
    # Tokenizing the data
    tokens = tokenizer(
        data['Text'].tolist(),
        max_length=max_length,
        padding='max_length',
        truncation=True,
        return_tensors="np"
    )
    
    return tokens['input_ids'], tokens['attention_mask']

# Maximum sequence length
max_length = 512

# Tokenize data and prepare attention masks
input_ids, attention_masks = tokenize_and_prepare_masks(data, max_length)
input_ids.shape, attention_masks.shape

((900, 512), (900, 512))

In [9]:
def generate_labels(text, summary, tokenizer, max_length):
    tokenized_text = tokenizer.encode(text, add_special_tokens=False)
    tokenized_summary = tokenizer.encode(summary, add_special_tokens=False)
    
    # Initialize labels with 0 (not in summary)
    labels = [0] * len(tokenized_text)
    
    # Set labels to 1 for tokens that appear in the summary
    summary_indices = [i for i, token in enumerate(tokenized_text) if token in tokenized_summary]
    for idx in summary_indices:
        labels[idx] = 1
    
    # Ensure labels match the required max_length
    if len(labels) > max_length:
        labels = labels[:max_length]
    else:
        labels += [0] * (max_length - len(labels))
    
    return labels

# Applying label creation across the dataset
data['labels'] = data.apply(lambda x: generate_labels(x['Text'], x['Short Summary'], tokenizer, max_length), axis=1)

# Convert labels list to appropriate format for training
labels = np.array(data['labels'].tolist())


Token indices sequence length is longer than the specified maximum sequence length for this model (615 > 512). Running this sequence through the model will result in indexing errors


In [10]:
from sklearn.model_selection import train_test_split

# Split input_ids and labels
x_train_ids, x_val_ids, y_train, y_val = train_test_split(
    input_ids, labels, test_size=0.1, random_state=42
)

# Split attention_masks using the same indices
_, x_val_masks = train_test_split(
    attention_masks, test_size=0.1, random_state=42
)

# Build and train model

In [11]:
from transformers import TFBertModel
from keras.layers import Input, Dense, Dropout, Attention
from keras.models import Model

def create_extractive_summarization_model(max_length):
    # Define input layers
    input_ids = Input(shape=(max_length,), dtype=tf.int32)
    attention_mask = Input(shape=(max_length,), dtype=tf.int32)
    
    # Load pre-trained BERT model
    bert_model = TFBertModel.from_pretrained('bert-base-uncased')
    
    # BERT-based encoder
    outputs = bert_model(input_ids, attention_mask=attention_mask)[0]
    
    # Attention mechanism
    attention_output = Attention()([outputs, outputs])
    
    # Dense layer
    dense_layer = Dense(units=max_length, activation='relu')(attention_output)
    
    # Output layer
    output = Dense(units=1, activation='sigmoid')(dense_layer)
    
    # Define model
    model = Model(inputs=[input_ids, attention_mask], outputs=output)
    
    return model

# Maximum sequence length
max_length = 192

# Create the model
model = create_extractive_summarization_model(max_length)

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Print model summary
model.summary()

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.seq_relationship.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.bias', 'cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.bias']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertModel for predictions w

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 192)]        0           []                               
                                                                                                  
 input_2 (InputLayer)           [(None, 192)]        0           []                               
                                                                                                  
 tf_bert_model_2 (TFBertModel)  TFBaseModelOutputWi  109482240   ['input_1[0][0]',                
                                thPoolingAndCrossAt               'input_2[0][0]']                
                                tentions(last_hidde                                               
                                n_state=(None, 192,                                           

In [ ]:
history = model.fit(
    [x_train_ids, _], y_train,
    validation_data=([x_val_ids, x_val_masks], y_val),
    epochs=3, batch_size=batch_size, callbacks=[tensorboard_callback]
)

In [ ]:
# Save the model
model_path = "./BERT MODIFIED/"
model.save(model_path+'model.h5')

# Test and Evaluate

In [13]:
import tensorflow as tf
from transformers import TFBertModel

# Register the custom object
custom_objects = {'TFBertModel': TFBertModel}

# Load the model with custom objects
model = tf.keras.models.load_model('./BERT MODIFIED/model.h5', custom_objects=custom_objects)

In [17]:
from rouge_score import rouge_scorer
import numpy as np  # Import numpy for averaging

def calculate_rouge_scores(references, hypotheses):
    # Initialize the RougeScorer with the desired metrics and a stemmer
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    scores = {
        'rouge1': [],
        'rouge2': [],
        'rougeL': []
    }

    # Score each pair of reference and hypothesis
    for ref, hyp in zip(references, hypotheses):
        score = scorer.score(ref, hyp)
        scores['rouge1'].append(score['rouge1'].fmeasure)
        scores['rouge2'].append(score['rouge2'].fmeasure)
        scores['rougeL'].append(score['rougeL'].fmeasure)

    # Calculate average scores using numpy to average the lists
    avg_scores = {key: np.mean(val) for key, val in scores.items()}
    return avg_scores


In [18]:
def preprocess_and_tokenize(text, tokenizer, max_length=512):
    # Tokenizing the single text input
    tokens = tokenizer(
        text,  # Direct text input
        max_length=max_length,
        padding='max_length',
        truncation=True,
        return_tensors="tf"  # Ensure this is set to TensorFlow tensors for the model compatibility
    )
    return tokens['input_ids'], tokens['attention_mask']


In [39]:
def generate_summary(input_ids, attention_mask, model, tokenizer):
    # Get model predictions
    predictions = model.predict([input_ids, attention_mask])[0]  # Assuming the model outputs logits

    # Apply threshold to get binary mask
    thresholded_prediction = (predictions > 0.5).astype(int)

    # Convert to flat numpy array if necessary
    thresholded_prediction = thresholded_prediction.flatten()

    # Extract the tokens that are part of the summary
    summary_tokens = [tok for idx, tok in enumerate(input_ids[0]) if thresholded_prediction[idx] == 1]

    # Decode the selected tokens to text
    summary_text = tokenizer.decode(summary_tokens)
    summary_text = summary_text.replace('[PAD]', '')
    summary_text = summary_text.replace('[UNK]', '')
    summary_text = summary_text.replace('#', '')
    summary_text = summary_text.replace('[CLS]', '')
    summary_text = summary_text.replace('[SEP]', '')
    return summary_text


In [40]:
def create_summary_from_text(text, tokenizer, model, max_length=512):
    # Preprocess and tokenize the input text
    input_ids, attention_mask = preprocess_and_tokenize(text, tokenizer, max_length)
    
    # Generate the summary
    summary = generate_summary(input_ids, attention_mask, model, tokenizer)
    
    return summary


In [41]:
i = 30
text = data['Text'][i]
orig_sum = data['Short Summary'][i]


user_input_text = text
# Assuming `model` and `tokenizer` are already loaded and available
summary_text = create_summary_from_text(user_input_text, tokenizer, model)
print("Actuall Text", text)
print("Generated Summary:", summary_text)
print("Orignal Summary", orig_sum)

1/1 [==============================] - 0s 447ms/step
Actuall Text اس پر منطقی قاعدے سے یہ اعتراض ہو سکتا ہے کہ رات کی تاریکی سب کے لئے یکساں ہوتی ہے پھر ایک خاص شخص کی رات سب سے زیادہ تاریک کیونکر ہو سکتی ہے؟ اور تمام کواکب ایسے اجرام ہیں جن کا وجود بغیر روشنی کے تصور میں نہیں آ سکتا۔ 
Generated Summary:  اس پر منطقی قاعدے سے یہ اعتراض ہو سکتا ہے کہ رات کی تاریکی سب کے ليے یکساں ہوتی ہے پھر ایک خاص شخص کی رات سب سے زیادہ تاریک کیونکر ہو سکتی ہے  اور تمام کواکب ایسے اجرام ہیں جن کا وجود بغیر روشنی کے تصور میں نہیں ا سکتا                                                                                                                                                                                                                                                                                                                                                       
Orignal Summary اس پر منطقی قاعدے سے یہ اعتراض ہو سکتا ہے کہ رات کی تاریکی سب کے لئے یکساں ہوتی ہے پھر ایک خاص شخص کی رات سب سے زی

In [42]:
orig = []
pred = []
for i in range(900,999):
    summary = create_summary_from_text(test_data['Text'][i], tokenizer,  model)
    orig.append(test_data['Short Summary'][i])
    pred.append(summary)

1/1 [==============================] - 0s 498ms/step


In [43]:
rouge_scores = calculate_rouge_scores(pred, orig)
print("ROUGE Scores:", rouge_scores)

ROUGE Scores: {'rouge1': 0.06552336552336552, 'rouge2': 0.018902309811400723, 'rougeL': 0.06552336552336552}
